# SLR-RAG : Step 1

## Package import

In [ ]:
# Import packages
%load_ext autoreload
%autoreload 2
import pandas as pd

from slrrag.querier_scopus import query_scopus

pd.__version__

## Inputs

### Query

In [ ]:
main_query = """
("risk assessment" OR "risk analysis") AND
("Offshore windfarm" OR "Offshore windmill" OR "Offshore wind turbine" OR "Offshore wind park" OR "OWF") AND
("Strategic" OR "Static" OR "Long-term" OR "Marine spatial planning" OR "Large scale") AND
("maritime" OR  "navigational" OR "nautical")
"""

### Reference papers
Represent them as (TITLE,DOI)
The notebook will try to do the merge based on the DOI first, and on the title otherwise if the paper has no DOI.

In [ ]:
from slrrag.settings import SchemaPapers

reference_papers = [
    (
        "A dynamic risk assessment method to address safety of navigation concerns around offshore renewable energy installations",
        "10.1177/1475090219837409",
    ),
    (
        "A Marine GIS-based Alert System to Prevent Vessels Collision with Offshore Platforms",
        "10.51400/2709-6998.2462",
    ),
]
df_reference_paper = pd.DataFrame(
    reference_papers,
    columns=[
        "title",
        "doi",
    ],
)
df_reference_paper = SchemaPapers.validate(df_reference_paper, lazy=True)
df_reference_paper

## Query databases

### Add scopus papers

In [ ]:
# You can edit your scopus API key in ~/.config/pybliometrics.cfg if needed
df_scopus = query_scopus(main_query)
df_scopus

In [ ]:
from slrrag.main import merge_papers

df_papers = merge_papers(df_reference_paper, df_scopus, "scopus")
df_papers

### Add OpenAlex papers

You will need the API key of

Make sure you have the API key of OpenAlex setup.

In [ ]:
from slrrag.querier_openalex import query_openalex

df_openalex = query_openalex(main_query)
df_openalex

In [ ]:
df_papers = merge_papers(df_papers, df_openalex, "openalex")
df_papers

### Add Lens

In [ ]:
from slrrag.main import print_lens_export_info
from slrrag.query_converter import convert_query_to_lens
from slrrag.settings import PATH_LENS_EXPORT

lens_export_file = PATH_LENS_EXPORT
lens_query = convert_query_to_lens(main_query)
print_lens_export_info(lens_query, lens_export_file)

In [ ]:
from slrrag.querier_lens import read_lens_export

df_lens = read_lens_export(lens_export_file)
df_lens

In [ ]:
df_papers = merge_papers(df_papers, df_lens, "lens")
df_papers

### Add dimensions

In [ ]:
from slrrag.main import print_dimensions_export
from slrrag.query_converter import query_to_dimensions_url
from slrrag.settings import PATH_DIMENSIONS_EXPORT

dimensions_url = query_to_dimensions_url(main_query)
export_path_dimensions = PATH_DIMENSIONS_EXPORT
print_dimensions_export(dimensions_url, export_path_dimensions)

In [ ]:
from slrrag.querier_dimensions import read_dimensions_export

df_dimensions = read_dimensions_export(export_path_dimensions)
df_dimensions

In [ ]:
df_papers = merge_papers(df_papers, df_dimensions, "dimensions")
df_papers

### Add useful columns

In [ ]:
from slrrag.main import complete_papers_df, show_merged_stats
from slrrag.settings import PATH_CSV_01_QUERY_RESULTS

databases_names = ["scopus", "openalex", "lens", "dimensions"]

df_papers = complete_papers_df(df_papers, database_names=databases_names)

df_papers.to_csv(PATH_CSV_01_QUERY_RESULTS)
print("Phase 1 results saved to :", PATH_CSV_01_QUERY_RESULTS)

## Papers Analysis

In [ ]:
# list of databases used
show_merged_stats(df_papers, databases_names)

#### Upset plot

In [ ]:
import pandas as pd

from slrrag.settings import PATH_CSV_01_QUERY_RESULTS

df_papers = pd.read_csv(PATH_CSV_01_QUERY_RESULTS)

df_papers.dtypes

In [ ]:
from slrrag.settings import PATH_UPSET_DISTINCT
from slrrag.toolbox.plotly_visualze import create_upset_plot

databases_names = ["scopus", "openalex", "lens", "dimensions"]

path_upset_distinct = PATH_UPSET_DISTINCT
fig_upset_distinct = create_upset_plot(
    df_papers, ["in_" + x for x in databases_names] + ["is_reference"], mode="distinct"
)
fig_upset_distinct.figure.write_html(path_upset_distinct)
print("Upset distinct plot saved to : ", path_upset_distinct)
fig_upset_distinct.figure

In [ ]:
from slrrag.settings import PATH_UPSET_INTERSECT

path_upset_intersect = PATH_UPSET_INTERSECT
fig_upset_distinct = create_upset_plot(
    df_papers, ["in_" + x for x in databases_names] + ["is_reference"], mode="intersect"
)
fig_upset_distinct.figure.write_html(path_upset_intersect)
print("Upset distinct plot saved to : ", path_upset_intersect)
fig_upset_distinct.figure